# PTQ (FLOAT16)

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택'

In [1]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


## Import Module

In [3]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [4]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 0s 0us/step


## Load Baseline Model for MNIST (CNN)

In [5]:
model = tf.keras.models.load_model('/content/drive/MyDrive/files/save/baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0

In [6]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9900000095367432


## Convert to TFLite model (Baseline model)

In [7]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_baseline_model = converter.convert()

In [8]:
litert_model_path = "/content/drive/MyDrive/files/save"

In [9]:
import pathlib

tflite_models_dir = pathlib.Path(litert_model_path)
tflite_models_dir.mkdir(exist_ok=True, parents=True)

tflite_baseline_model_file = tflite_models_dir/"mnist_baseline_model.tflite"
size_baseline_model = tflite_baseline_model_file.write_bytes(tflite_baseline_model)

In [10]:
interpreter_base = tf.lite.Interpreter(model_content=tflite_baseline_model)
interpreter_base.allocate_tensors()

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [11]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_ptq_float16_model = converter.convert()

tflite_ptq_float16_model_file = tflite_models_dir/"mnist_ptq_float16_model.tflite"
size_ptq_float16_model = tflite_ptq_float16_model_file.write_bytes(tflite_ptq_float16_model)

In [12]:
print("Size of Baseline LiteRT Model file : {}".format(size_baseline_model))
print("Size of PTQ(FLOAT16) LiteRT Model file : {}".format(size_ptq_float16_model))

Size of Baseline LiteRT Model file : 234392
Size of PTQ(FLOAT16) LiteRT Model file : 120240


## PTQ (FLOAT16) model의 input/output dtype 확인

In [13]:
interpreter_ptq_float16 = tf.lite.Interpreter(model_content=tflite_ptq_float16_model)
interpreter_ptq_float16.allocate_tensors()

input_dtype = interpreter_ptq_float16.get_input_details()[0]['dtype']
output_dtype = interpreter_ptq_float16.get_output_details()[0]['dtype']

print('input dtype: ', input_dtype)
print('output dtype: ', output_dtype)

input dtype:  <class 'numpy.float32'>
output dtype:  <class 'numpy.float32'>


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## 추론 실행 코드

In [14]:
test_image = np.expand_dims(test_images[0], axis=0)

input_index = interpreter_ptq_float16.get_input_details()[0]["index"]
output_index = interpreter_ptq_float16.get_output_details()[0]["index"]

interpreter_ptq_float16.set_tensor(input_index, test_image)

interpreter_ptq_float16.invoke()

predictions = interpreter_ptq_float16.get_tensor(output_index)

print(predictions)
print(np.argmax(predictions))
print(test_labels[0])

[[3.2287586e-12 1.0226030e-09 1.7879490e-09 5.3659939e-09 2.3751264e-16
  4.7981282e-11 5.8041480e-20 1.0000000e+00 5.9867666e-10 8.7774588e-09]]
7
7


## Test data 기반 accuracy 평가

In [15]:
def evaluate_model(interpreter):
  input_details = interpreter.get_input_details()
  output_details = interpreter.get_output_details()

  input_index = input_details[0]["index"]
  output_index = output_details[0]["index"]

  prediction_digits = []
  for test_image in test_images:
    test_image = np.expand_dims(test_image, axis=0)
    interpreter.set_tensor(input_index, test_image)

    interpreter.invoke()

    output = interpreter.get_tensor(output_index)
    digit = np.argmax(output)
    prediction_digits.append(digit)

  accurate_count = 0
  for index in range(len(prediction_digits)):
    if prediction_digits[index] == test_labels[index]:
      accurate_count += 1
  accuracy = accurate_count * 1.0 / len(prediction_digits)

  return accuracy

In [16]:
print(evaluate_model(interpreter_base))
print(evaluate_model(interpreter_ptq_float16))

0.99
0.99
